# Experimental Validation Demo

Compare Sicifus predictions to experimentally measured ΔΔG values from the literature.

This notebook demonstrates:
1. **Benchmark dataset** - Load experimental data from published studies
2. **Prediction workflow** - Download structures, run calculations
3. **Performance metrics** - R², RMSE, MAE, correlations
4. **Publication plots** - Scatter plots with statistics, error distributions

**Dataset:** 13 well-characterized mutations from Barnase, T4 Lysozyme, and Chymotrypsin Inhibitor.

**Sources:**
- Serrano et al. (1992) - Barnase mutations
- Matsumura et al. (1988) - T4 Lysozyme core mutations
- Eriksson et al. (1992) - T4 Lysozyme cavity mutations
- Jackson et al. (1993) - Chymotrypsin Inhibitor 2

**Note:** By default, this notebook uses **mock predictions** for fast demonstration (~10 seconds).
Set `use_real_predictions=True` to download PDBs and run actual calculations (10-30 minutes, requires OpenMM).

## Setup

In [ ]:
import os
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

# For real predictions (requires OpenMM)
try:
    from sicifus import MutationEngine
    OPENMM_AVAILABLE = True
    print("✅ OpenMM available - can run real predictions")
except ImportError:
    OPENMM_AVAILABLE = False
    print("⚠️  OpenMM not available - will use mock predictions")
    print("   Install with: pip install sicifus[energy]")

## Step 1: Load Benchmark Dataset

Load experimental ΔΔG values from published mutagenesis studies.

**Dataset columns:**
- `protein` - Protein name
- `pdb_id` - PDB structure code
- `mutation` - Mutation in format `WtPositionMut` (e.g., `F13A`)
- `chain` - Chain identifier
- `experimental_ddg` - Measured ΔΔG in kcal/mol
- `reference` - Literature source

In [ ]:
# Embedded benchmark dataset (real experimental data)
EXAMPLE_DATASET = """
protein,pdb_id,mutation,chain,experimental_ddg,reference
Barnase,1BNI,A24G,A,0.61,Serrano1992
Barnase,1BNI,G23A,A,-0.44,Serrano1992
Barnase,1BNI,V36L,A,-0.23,Serrano1992
Barnase,1BNI,F56A,A,1.77,Serrano1992
Barnase,1BNI,Y24A,A,1.02,Serrano1992
T4_Lysozyme,2LZM,M6A,A,2.1,Matsumura1988
T4_Lysozyme,2LZM,I3A,A,1.94,Matsumura1988
T4_Lysozyme,2LZM,L99A,A,4.0,Eriksson1992
T4_Lysozyme,2LZM,V87A,A,1.36,Matsumura1988
T4_Lysozyme,2LZM,I29A,A,1.52,Matsumura1988
Chymotrypsin_Inhibitor,2CI2,L49A,A,1.8,Jackson1993
Chymotrypsin_Inhibitor,2CI2,V47A,A,0.72,Jackson1993
Chymotrypsin_Inhibitor,2CI2,I57A,A,1.98,Jackson1993
"""

# Load dataset
dataset_df = pl.read_csv(EXAMPLE_DATASET.strip().encode())

# Create output directory
os.makedirs("validation_data", exist_ok=True)
dataset_df.write_csv("validation_data/validation_dataset.csv")

print(f"Loaded {len(dataset_df)} mutations from {dataset_df['protein'].n_unique()} proteins\n")
dataset_df

In [ ]:
# Summary statistics
print("Dataset Summary:")
print(f"  Proteins: {dataset_df['protein'].unique().to_list()}")
print(f"  PDB codes: {dataset_df['pdb_id'].unique().to_list()}")
print(f"  Experimental ΔΔG range: [{dataset_df['experimental_ddg'].min():.2f}, {dataset_df['experimental_ddg'].max():.2f}] kcal/mol")
print(f"  Mean experimental ΔΔG: {dataset_df['experimental_ddg'].mean():.2f} kcal/mol")

## Step 2: Run Predictions

**Two modes:**

### Mock Predictions (default, fast)
- Uses correlated noise to simulate realistic prediction errors
- Runs in ~1 second
- Good for testing workflow and visualization

### Real Predictions (requires OpenMM)
- Downloads PDB structures from RCSB
- Repairs structures with PDBFixer
- Runs OpenMM minimization
- Takes 10-30 minutes (13 mutations × 3 runs each)

**To use real predictions:** Set `use_real_predictions=True` below.

### Option A: Mock Predictions (Fast Demo)

In [ ]:
# Mock predictions with realistic correlated noise
np.random.seed(42)
experimental = dataset_df['experimental_ddg'].to_numpy()

# Simulate typical force field errors:
# - Random noise (σ ~ 0.6 kcal/mol)
# - Systematic underestimation of large destabilizing mutations
noise = np.random.normal(0, 0.6, len(dataset_df))
bias = np.where(experimental > 2.0, -0.5, 0.0)
mock_predictions = experimental + noise + bias

# Add to dataframe
results_df = dataset_df.with_columns(pl.Series("predicted_ddg", mock_predictions))

print("Using mock predictions (for fast demo)")
print("\nResults preview:")
results_df.select(["mutation", "experimental_ddg", "predicted_ddg"])

### Option B: Real Predictions (Slow, requires OpenMM)

**Uncomment and run the cell below to use real predictions instead.**

In [ ]:
# # Real predictions - WARNING: Takes 10-30 minutes!
# if not OPENMM_AVAILABLE:
#     print("ERROR: OpenMM not installed. Install with: pip install sicifus[energy]")
# else:
#     import urllib.request
#     
#     engine = MutationEngine()
#     predictions = []
#     
#     print("Running real predictions (this will take 10-30 minutes)...\n")
#     
#     # Process each PDB structure
#     for pdb_id in dataset_df['pdb_id'].unique():
#         print(f"\n{pdb_id}:")
#         
#         # Download and clean structure
#         pdb_path = Path(f"validation_data/{pdb_id}_clean.pdb")
#         if not pdb_path.exists():
#             print(f"  Downloading from RCSB...")
#             url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
#             raw_path = Path(f"validation_data/{pdb_id}_raw.pdb")
#             urllib.request.urlretrieve(url, raw_path)
#             
#             # Clean: remove waters and heteroatoms
#             with open(raw_path, 'r') as f_in, open(pdb_path, 'w') as f_out:
#                 for line in f_in:
#                     if line.startswith('ATOM'):
#                         f_out.write(line)
#         else:
#             print(f"  Using cached structure")
#         
#         # Repair structure
#         print(f"  Repairing structure...")
#         repaired_pdb = engine.repair(str(pdb_path))
#         repaired_path = pdb_path.parent / f"{pdb_id}_repaired.pdb"
#         with open(repaired_path, 'w') as f:
#             f.write(repaired_pdb)
#         
#         # Run mutations
#         pdb_mutations = dataset_df.filter(pl.col('pdb_id') == pdb_id)
#         for row in pdb_mutations.iter_rows(named=True):
#             mutation = row['mutation']
#             print(f"  {mutation}...", end=" ", flush=True)
#             
#             result = engine.mutate(
#                 str(repaired_path),
#                 [mutation],
#                 chain=row['chain'],
#                 n_runs=3,
#                 max_iterations=1000,
#                 keep_statistics=True
#             )
#             
#             predicted = result.ddg_mean[mutation] if result.ddg_mean else result.ddg[mutation]
#             predictions.append({
#                 'protein': row['protein'],
#                 'pdb_id': pdb_id,
#                 'mutation': mutation,
#                 'chain': row['chain'],
#                 'experimental_ddg': row['experimental_ddg'],
#                 'predicted_ddg': predicted,
#                 'reference': row['reference']
#             })
#             
#             print(f"Predicted: {predicted:+.2f} (Exp: {row['experimental_ddg']:+.2f})")
#     
#     # Replace mock results with real predictions
#     results_df = pl.DataFrame(predictions)
#     print("\nReal predictions complete!")

In [ ]:
# Save results
results_df.write_csv("validation_data/prediction_results.csv")
print("Results saved to validation_data/prediction_results.csv")

## Step 3: Calculate Performance Metrics

Evaluate prediction quality using standard regression metrics:

- **R² (R-squared)** - Coefficient of determination (0-1, higher is better)
- **RMSE** - Root mean squared error (kcal/mol, lower is better)
- **MAE** - Mean absolute error (kcal/mol, lower is better)
- **Pearson r** - Linear correlation (-1 to 1)
- **Spearman ρ** - Rank correlation (more robust to outliers)

In [ ]:
# Extract arrays
experimental = results_df['experimental_ddg'].to_numpy()
predicted = results_df['predicted_ddg'].to_numpy()

# Calculate metrics
r_squared = stats.pearsonr(experimental, predicted)[0] ** 2
rmse = np.sqrt(np.mean((predicted - experimental) ** 2))
mae = np.mean(np.abs(predicted - experimental))
pearson_r, pearson_p = stats.pearsonr(experimental, predicted)
spearman_rho, spearman_p = stats.spearmanr(experimental, predicted)

# Display
print("="*60)
print("PERFORMANCE METRICS")
print("="*60)
print(f"Sample size:       {len(experimental)} mutations")
print(f"R² (R-squared):    {r_squared:.3f}")
print(f"RMSE:              {rmse:.2f} kcal/mol")
print(f"MAE:               {mae:.2f} kcal/mol")
print(f"Pearson r:         {pearson_r:.3f} (p = {pearson_p:.2e})")
print(f"Spearman ρ:        {spearman_rho:.3f} (p = {spearman_p:.2e})")
print("="*60)

In [ ]:
# Interpretation
print("\nINTERPRETATION:")

if r_squared > 0.5:
    print(f"✅ Good correlation (R² = {r_squared:.3f})")
elif r_squared > 0.3:
    print(f"⚠️  Moderate correlation (R² = {r_squared:.3f})")
else:
    print(f"❌ Weak correlation (R² = {r_squared:.3f})")

if rmse < 1.0:
    print(f"✅ Low RMSE ({rmse:.2f} kcal/mol)")
elif rmse < 2.0:
    print(f"⚠️  Moderate RMSE ({rmse:.2f} kcal/mol)")
else:
    print(f"❌ High RMSE ({rmse:.2f} kcal/mol)")

print("\nContext: State-of-the-art computational methods typically achieve:")
print("  - R² ~ 0.4-0.7 for diverse mutation sets")
print("  - RMSE ~ 1.0-1.5 kcal/mol")
print("  (Experimental uncertainty ~ 0.5-1.0 kcal/mol)")

## Step 4: Visualization

### Plot 1: Predicted vs Experimental Scatter

In [ ]:
# Create publication-quality scatter plot
fig, ax = plt.subplots(figsize=(10, 10))

# Scatter plot
ax.scatter(experimental, predicted, alpha=0.6, s=150, 
           edgecolors='black', linewidth=1.5, c='steelblue')

# Perfect prediction line (y = x)
min_val = min(experimental.min(), predicted.min()) - 0.5
max_val = max(experimental.max(), predicted.max()) + 0.5
ax.plot([min_val, max_val], [min_val, max_val], 
        'k--', alpha=0.5, linewidth=2, label='Perfect prediction')

# Linear regression fit
slope, intercept = np.polyfit(experimental, predicted, 1)
fit_x = np.linspace(min_val, max_val, 100)
fit_y = slope * fit_x + intercept
ax.plot(fit_x, fit_y, 'r-', alpha=0.7, linewidth=2,
        label=f'Linear fit (y = {slope:.2f}x + {intercept:+.2f})')

# Labels
ax.set_xlabel('Experimental ΔΔG (kcal/mol)', fontsize=14, fontweight='bold')
ax.set_ylabel('Predicted ΔΔG (kcal/mol)', fontsize=14, fontweight='bold')
ax.set_title('Sicifus Validation: Predicted vs Experimental',
             fontsize=16, fontweight='bold', pad=20)

# Statistics text box
stats_text = (
    f"n = {len(experimental)}\n"
    f"R² = {r_squared:.3f}\n"
    f"RMSE = {rmse:.2f} kcal/mol\n"
    f"MAE = {mae:.2f} kcal/mol\n"
    f"Pearson r = {pearson_r:.3f}\n"
    f"Spearman ρ = {spearman_rho:.3f}"
)

ax.text(0.05, 0.95, stats_text, transform=ax.transAxes,
        fontsize=12, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.9, pad=0.8))

# Formatting
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(loc='lower right', fontsize=12)
ax.set_aspect('equal', adjustable='box')

plt.tight_layout()
plt.savefig("validation_plot.png", dpi=300, bbox_inches='tight')
print("Saved to validation_plot.png")
plt.show()

### Plot 2: Error Distribution Analysis

In [ ]:
# Calculate errors
errors = predicted - experimental
abs_errors = np.abs(errors)

# Two-panel plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Error histogram
ax1.hist(errors, bins=15, alpha=0.7, color='steelblue', edgecolor='black', linewidth=1.5)
ax1.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero error')
ax1.axvline(np.mean(errors), color='orange', linestyle='-', linewidth=2,
            label=f'Mean error: {np.mean(errors):.2f}')
ax1.set_xlabel('Prediction Error (kcal/mol)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax1.set_title('Distribution of Prediction Errors', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Panel 2: Error vs magnitude
abs_experimental = np.abs(experimental)
ax2.scatter(abs_experimental, abs_errors, alpha=0.6, s=150,
            edgecolors='black', linewidth=1.5, color='coral')
ax2.set_xlabel('|Experimental ΔΔG| (kcal/mol)', fontsize=12, fontweight='bold')
ax2.set_ylabel('|Prediction Error| (kcal/mol)', fontsize=12, fontweight='bold')
ax2.set_title('Absolute Error vs Mutation Magnitude', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Add MAE line
ax2.axhline(mae, color='red', linestyle='--', linewidth=2,
            label=f'MAE: {mae:.2f}')
ax2.legend(fontsize=11)

plt.tight_layout()
plt.savefig("error_distribution.png", dpi=300, bbox_inches='tight')
print("Saved to error_distribution.png")
plt.show()

**Interpretation:**

- **Left panel:** Symmetric error distribution (centered near zero) indicates no systematic bias
- **Right panel:** If errors increase with mutation magnitude, the method struggles with large effects
  - Common issue: Force fields underestimate large destabilizing mutations (e.g., core deletions)

## Step 5: Per-Protein Breakdown

Check if performance varies across different proteins.

In [ ]:
# Calculate metrics per protein
for protein in results_df['protein'].unique():
    protein_df = results_df.filter(pl.col('protein') == protein)
    exp = protein_df['experimental_ddg'].to_numpy()
    pred = protein_df['predicted_ddg'].to_numpy()
    
    if len(exp) > 2:  # Need at least 3 points for correlation
        r2 = stats.pearsonr(exp, pred)[0] ** 2
        rmse_p = np.sqrt(np.mean((pred - exp) ** 2))
        print(f"{protein}:")
        print(f"  n = {len(exp)}, R² = {r2:.3f}, RMSE = {rmse_p:.2f} kcal/mol")
    else:
        print(f"{protein}: Too few points for correlation")

## Extending to Larger Datasets

To validate on hundreds or thousands of mutations:

### 1. Download a larger benchmark

**ProTherm** (10,000+ mutations): [https://web.iitm.ac.in/bioinfo2/prothermdb/](https://web.iitm.ac.in/bioinfo2/prothermdb/)

**SKEMPI 2.0** (binding affinity changes): [https://life.bsc.es/pid/skempi2](https://life.bsc.es/pid/skempi2)

**S669** (stability changes): Various publications

### 2. Format as CSV

```csv
protein,pdb_id,mutation,chain,experimental_ddg,reference
MyProtein,1ABC,F13A,A,1.5,Author2023
...
```

### 3. Run batch predictions

```python
# Load your dataset
large_dataset = pl.read_csv("my_benchmark.csv")

# Run predictions (can take hours for large datasets)
results = predict_ddg_batch(large_dataset, use_real_predictions=True)

# Analyze
metrics = calculate_metrics(results['experimental_ddg'], results['predicted_ddg'])
```

### 4. Consider stratification

Performance can vary by:
- **Mutation type** (hydrophobic→hydrophobic vs charged→uncharged)
- **Location** (core vs surface vs interface)
- **Protein family** (all-α vs all-β vs mixed)

Analyze each subset separately for deeper insights.

## Summary

This notebook demonstrated:

1. ✅ **Benchmark loading** - Experimental data from literature
2. ✅ **Automated predictions** - Download, repair, minimize, calculate
3. ✅ **Rigorous metrics** - R², RMSE, MAE, correlations
4. ✅ **Publication plots** - Scatter plots and error analysis
5. ✅ **Scalability** - Framework for large-scale validation

### Key Takeaways

- **Computational prediction is challenging** - Even state-of-the-art methods have RMSE ~ 1-1.5 kcal/mol
- **Experimental uncertainty exists** - Measured ΔΔG values can vary 0.5-1.0 kcal/mol between labs
- **Trends matter more than absolutes** - Focus on correctly ranking mutations (Spearman ρ)
- **Know your limitations** - Large cavity-creating mutations are harder to predict

### Next Steps

- Validate on your own experimental data
- Compare to other methods (FoldX, Rosetta, machine learning)
- Tune parameters (minimization steps, force field, protonation)
- Check out `mutation_analysis_demo.ipynb` for more analysis types

## Your Experiments

Use the cells below to validate with your own data:

In [ ]:
# Your code here
